# Notebook 00 — Descarga de Datos Crudos

## Objetivo

Descarga los tres datasets crudos del proyecto y los guarda en `data/raw/`.

**Idempotencia**: `download_sp500()` y `download_macro()` verifican si el CSV ya existe
en `data/raw/` antes de pegarle a la API. Si ya está descargado, simplemente lo cargan.
Esto significa que correr este notebook de nuevo **no vuelve a descargar nada** a menos
que borres los archivos de `data/raw/` manualmente.

In [1]:
import sys
import os
sys.path.insert(0, "..")

from src import data_loader, utils, features

utils.set_plot_style()

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

START = "2008-01-01"
END   = "2026-06-18"

## Descarga de precios S&P 500

Ticker `^GSPC` via `yfinance`. Guarda en `data/raw/sp500_raw.csv`.

In [2]:
sp500_path = os.path.join(RAW_DIR, "sp500_raw.csv")
sp500 = data_loader.download_sp500(START, END, sp500_path)

print(f"S&P 500: {sp500.shape} | {sp500.index[0].date()} → {sp500.index[-1].date()}")

[data_loader] sp500_raw.csv ya existe en ../data/raw/sp500_raw.csv, se carga sin descargar.
S&P 500: (4644, 5) | 2008-01-02 → 2026-06-17


## Descarga de indicadores macroeconómicos (FRED)

Series descargadas: VIX (`VIXCLS`), spread de tasas (`T10Y2Y`), Fed Funds Rate
(`FEDFUNDS`), inflación (`CPIAUCSL`) y desempleo (`UNRATE`).

La API key se lee de `.env` dentro de `download_macro()` — no se expone en el notebook.
Guarda en `data/raw/macro_fred.csv`.

In [3]:
macro_path = os.path.join(RAW_DIR, "macro_fred.csv")
macro = data_loader.download_macro(START, END, macro_path)

print(f"Macro FRED: {macro.shape} | {macro.index[0].date()} → {macro.index[-1].date()}")

[data_loader] macro_fred.csv ya existe en ../data/raw/macro_fred.csv, se carga sin descargar.
Macro FRED: (4881, 8) | 2008-01-01 → 2026-06-18


## Índice de riesgo geopolítico (GPR)

Agrega GPRD, GPRD_ACT y GPRD_THREAT (Caldara & Iacoviello) a `macro_fred.csv` y
`macro_fred_production.csv`, vía `features.add_geopolitical_risk_index_macro()`.
Fuente: `matteoiacoviello.com/gpr_files/data_gpr_daily_recent.xls`, serie diaria
que se actualiza los lunes.

**Pendiente**: estos valores son crudos (fecha de observación, sin shiftear a
fecha de publicación real). El shift semanal análogo al de CPI/UNRATE todavía
no está implementado en `add_macro_features()` — no usar estas columnas como
feature hasta resolver eso.

In [4]:
features.add_geopolitical_risk_index_macro()

# Recargar para confirmar que las columnas quedaron
macro_gpr = data_loader.load_macro(RAW_DIR)
print(f"Macro FRED + GPR: {macro_gpr.shape} | columnas: {list(macro_gpr.columns)}")

[gpr] Descargando https://www.matteoiacoviello.com/gpr_files/data_gpr_daily_recent.xls ...
[gpr] GPR descargado: (15163, 3) | 1985-01-01 -> 2026-07-07
[gpr] /Users/nacho/Documents/Ingenieria en Inteligencia Artificial/3er Año/Primer cuatrimestre/Aprendizaje Automatico y Aprendizaje Profundo/Trabajos practicos/Trabajo Practico Final/Predictor-de-direcci-n-del-S-P-500/notebooks/../src/../data/raw/macro_fred.csv ya tiene columnas GPR, se sobreescriben.
[gpr] /Users/nacho/Documents/Ingenieria en Inteligencia Artificial/3er Año/Primer cuatrimestre/Aprendizaje Automatico y Aprendizaje Profundo/Trabajos practicos/Trabajo Practico Final/Predictor-de-direcci-n-del-S-P-500/notebooks/../src/../data/raw/macro_fred.csv: (4881, 5) -> (4881, 8) | filas sin match GPR: 0
Macro FRED + GPR: (4881, 8) | columnas: ['vix', 't10y2y', 'fedfunds', 'cpi', 'unrate', 'GPRD', 'GPRD_ACT', 'GPRD_THREAT']


## Noticias financieras (Kaggle)

Este dataset **no se descarga via API** — es un CSV estático de Kaggle (19,127 headlines,
2008–2024) que debe colocarse manualmente en `data/raw/sp500_news.csv`. Por eso no existe
un `download_news()` en `data_loader.py`, solo `load_news()`.

Si el archivo no está presente, hay que bajarlo de Kaggle antes de continuar.

In [5]:
news_path = os.path.join(RAW_DIR, "sp500_news_full.csv")
if not os.path.exists(news_path):
    raise FileNotFoundError(
        f"No se encontró {news_path}.\n"
        "Este dataset viene de Kaggle y no se descarga via API: "
        "hay que colocarlo manualmente en data/raw/sp500_news_full.csv."
    )

news = data_loader.load_news(RAW_DIR)
print(f"Noticias: {news.shape} | {news['Date'].min().date()} → {news['Date'].max().date()}")

Noticias: (29323, 2) | 2008-01-02 → 2026-06-18


## Resumen estadístico de los datos crudos

Verificación rápida de tipos de datos, nulos y shape de los tres datasets descargados.

In [6]:
utils.dataset_summary("S&P 500 Precios", sp500)
utils.dataset_summary("Macro FRED", macro)
utils.dataset_summary("Noticias Financieras", news)


  S&P 500 Precios  |  4,644 filas  x  5 columnas


,dtype,non_null,null,null_%,unique
Close,float64,4644,0,0.0,4621
High,float64,4644,0,0.0,4608
Low,float64,4644,0,0.0,4614
Open,float64,4644,0,0.0,4612
Volume,int64,4644,0,0.0,4574



── Muestra aleatoria (5 filas) ──


,Close,High,Low,Open,Volume
Date,,,,,
2013-10-29,1771.949951,1772.089966,1762.930054,1762.930054,3358460000
2015-03-03,2107.780029,2115.760010,2098.260010,2115.760010,3262300000
2019-03-18,2832.939941,2835.409912,2821.989990,2822.610107,3620770000
2008-11-28,896.239990,896.250000,881.210022,886.890015,2740860000
2022-09-15,3901.350098,3959.139893,3888.280029,3932.409912,4441830000



  Macro FRED  |  4,881 filas  x  8 columnas


...

,vix,t10y2y,fedfunds,cpi,unrate,GPRD,GPRD_ACT,GPRD_THREAT
Date,,,,,,,,
2018-03-13,16.35,0.58,1.51,249.577,4.0,119.587288,63.583565,145.846176
2010-09-28,22.60,2.11,0.19,218.275,9.5,89.327667,81.631645,99.038338
2009-02-13,42.93,1.92,0.22,212.705,8.3,57.332905,69.857857,47.674107
2021-08-31,16.48,1.10,0.09,272.676,5.1,203.012360,108.598160,288.214142
2023-09-21,17.54,-0.63,5.33,307.276,3.7,162.913162,108.274460,225.778992



  Noticias Financieras  |  29,323 filas  x  2 columnas


,dtype,non_null,null,null_%,unique
Title,str,29323,0,0.0,29215
Date,datetime64[us],29323,0,0.0,4255



── Muestra aleatoria (5 filas) ──


,Title,Date
5222,Stock market compared to PMI,2017-05-03
18795,Trading stocks all day and all night might be ...,2024-05-20
9805,US STOCKS-Wall Street rally fizzles out as Eve...,2021-09-24
14077,Q1 2023 Earnings Update: Better than Feared Un...,2023-05-15
4974,Here Are the Best and Worst Performing Assets ...,2016-12-28


### Limpieza
El dataset de noticias financieras y el de los precios del S&P 500 no presentaron valores nulos. Por otro lado, el Macro FRED presentó 206 correspondientes a días sin dato en VIX, los cuales serán eliminados con `dropna()`. 